# CS535 Project 3 — DeepNeoAG-Lite: PLM + Multi-window CNN

This notebook connects the IMDb sequence-learning project to the published DeepNeoAG research direction.

The public DeepNeoAG repository reports 7,012 sequences after <30% similarity filtering and length <35: 302 neoantigens and 6,710 other peptides, split 80/20 into 5,608 train and 1,404 test sequences.

For classroom speed, we use a small ESM-2 checkpoint first. For paper-level reproduction, use the public repository's documented feature workflow (ProtTrans/TAPE/ESM) and exact split.

In [ ]:
!pip -q install fair-esm scikit-learn pandas matplotlib

In [ ]:
import os,re,random,subprocess
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,balanced_accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,matthews_corrcoef,confusion_matrix
import esm
SEED=535
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu');FAST_MODE=True
print(DEVICE)

## 1. Clone the public repository and inspect the actual files

Never assume a file layout from memory. Inspect the repository before deciding how to parse labels and splits.

In [ ]:
REPO=Path('/content/DeepNeoAG')
if not REPO.exists(): subprocess.run(['git','clone','-q','https://github.com/B1607/DeepNeoAG.git',str(REPO)],check=True)
for p in list(REPO.rglob('*'))[:120]: print(p.relative_to(REPO))

## 2. Published dataset counts and why they matter

In [ ]:
published=pd.DataFrame({'class':['Neoantigen','Other','Total'],'initial':[671,32206,32877],'<30% similarity':[303,6717,7019],'length<35':[302,6710,7012],'train':[241,5367,5608],'test':[61,1343,1404]})
display(published)
print('positive fraction after filtering:',302/7012)

## 3. Parse FASTA files conservatively

The public repository contains `fasta`, `data`, and `dataset` directories. This helper lists candidate files and parses FASTA records. For formal reproduction, use the repository's documented Train/Test files rather than inventing a new random split.

In [ ]:
def parse_fasta(path):
    rec=[];name=None;seq=[]
    with open(path,encoding='utf-8',errors='ignore') as f:
        for line in f:
            line=line.strip()
            if not line:continue
            if line.startswith('>'):
                if name is not None:rec.append((name,''.join(seq)))
                name=line[1:];seq=[]
            else:seq.append(line)
    if name is not None:rec.append((name,''.join(seq)))
    return rec
fastas=[]
for ext in ['*.fa','*.fasta','*.faa','*.fas']:
    fastas.extend(REPO.rglob(ext))
print('FASTA count',len(fastas))
for p in fastas[:80]:print(p.relative_to(REPO),len(parse_fasta(p)))

## 4. Data adapter

Because repositories can evolve, this notebook first searches for explicit `sequence,label` tables. If none exist, it uses file-path keywords only as a classroom fallback and prints the resulting counts for inspection. Do not use heuristic labels in a paper.

In [ ]:
def discover(repo):
    for p in list(repo.rglob('*.csv'))+list(repo.rglob('*.tsv')):
        try:
            d=pd.read_csv(p,sep='\t' if p.suffix=='.tsv' else ',');low={x.lower():x for x in d.columns}
            sc=next((low[k] for k in low if k in ['sequence','seq','peptide','epitope']),None)
            lc=next((low[k] for k in low if k in ['label','class','target','y']),None)
            if sc and lc:
                o=d[[sc,lc]].copy();o.columns=['sequence','label'];return o,str(p.relative_to(repo))
        except Exception:pass
    rows=[]
    for p in fastas:
        s=str(p).lower();lab=None
        if any(k in s for k in ['neo','positive','pos']):lab=1
        elif any(k in s for k in ['other','negative','neg']):lab=0
        if lab is None:continue
        for h,seq in parse_fasta(p):
            seq=re.sub('[^ACDEFGHIKLMNPQRSTVWY]','',seq.upper())
            if 1<=len(seq)<35:rows.append((seq,lab,str(p.relative_to(repo))))
    return (pd.DataFrame(rows,columns=['sequence','label','source']),'heuristic FASTA fallback') if rows else (None,None)
df,source=discover(REPO);print('source=',source)
if df is not None:
    df['label']=pd.to_numeric(df['label'],errors='coerce');df=df.dropna(subset=['sequence','label']);df=df[df.label.isin([0,1])].copy();df.label=df.label.astype(int)
    print(df.shape);display(df.head());print(df.label.value_counts())

## 5. If the exact published split can be identified, use it

The fallback stratified split below is for teaching only. In a research reproduction, replace it with the repository's exact 80/20 files.

In [ ]:
if df is None:
    raise RuntimeError('No labeled data discovered. Inspect repository and point the notebook to exact published files.')
train_df,temp=train_test_split(df,test_size=.30,stratify=df.label,random_state=SEED)
val_df,test_df=train_test_split(temp,test_size=.50,stratify=temp.label,random_state=SEED)
if FAST_MODE:
    def cap(d,nneg):
        p=d[d.label==1];n=d[d.label==0].sample(min(nneg,(d.label==0).sum()),random_state=SEED);return pd.concat([p,n]).sample(frac=1,random_state=SEED).reset_index(drop=True)
    train_df=cap(train_df,1200);val_df=cap(val_df,400);test_df=cap(test_df,400)
print(train_df.label.value_counts(),val_df.label.value_counts(),test_df.label.value_counts())

## 6. Order-free baseline: amino-acid composition + logistic regression

This is the biological analogue of TF-IDF/logistic regression as a simple baseline. It ignores sequence order.

In [ ]:
AA='ACDEFGHIKLMNPQRSTVWY';aa2i={a:i for i,a in enumerate(AA)}
def comp(seq):
    v=np.zeros(20,dtype=float)
    for a in seq:
        if a in aa2i:v[aa2i[a]]+=1
    return v/max(1,len(seq))
Xtr=np.stack([comp(s) for s in train_df.sequence]);Xva=np.stack([comp(s) for s in val_df.sequence])
lr=LogisticRegression(max_iter=1000,class_weight='balanced').fit(Xtr,train_df.label)
p=lr.predict_proba(Xva)[:,1];print('AUPRC',average_precision_score(val_df.label,p),'MCC',matthews_corrcoef(val_df.label,p>=.5))

## 7. One-hot MCNN baseline

In [ ]:
MAX_LEN=35
def onehot(seq):
    x=torch.zeros(MAX_LEN,20)
    for i,a in enumerate(seq[:MAX_LEN]):
        if a in aa2i:x[i,aa2i[a]]=1
    return x
class PepDS(Dataset):
    def __init__(self,d):self.d=d.reset_index(drop=True)
    def __len__(self):return len(self.d)
    def __getitem__(self,i):r=self.d.iloc[i];return onehot(r.sequence),torch.tensor(float(r.label))
def loader(d,bs=64,shuffle=False):return DataLoader(PepDS(d),batch_size=bs,shuffle=shuffle)
ohtr,ohva,ohte=loader(train_df,64,True),loader(val_df),loader(test_df)
class OneHotMCNN(nn.Module):
    def __init__(self,windows=(2,4,8,16),filters=64,drop=.3):
        super().__init__();self.convs=nn.ModuleList([nn.Conv1d(20,filters,k) for k in windows]);self.drop=nn.Dropout(drop);self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,x):
        x=x.transpose(1,2);feats=[F.adaptive_max_pool1d(F.relu(c(x)),1).squeeze(-1) for c in self.convs];return self.fc(self.drop(torch.cat(feats,1))).squeeze(1)

In [ ]:
def mets(y,p):
    pred=(p>=.5).astype(int);tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(accuracy=accuracy_score(y,pred),balanced_accuracy=balanced_accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),specificity=tn/(tn+fp) if tn+fp else np.nan,f1=f1_score(y,pred,zero_division=0),mcc=matthews_corrcoef(y,pred),roc_auc=roc_auc_score(y,p),pr_auc=average_precision_score(y,p))
def evaluate(m,ld):
    m.eval();ys=[];ps=[]
    with torch.no_grad():
        for x,y in ld:
            p=torch.sigmoid(m(x.to(DEVICE))).cpu().numpy();ys.extend(y.numpy());ps.extend(p)
    return np.array(ys),np.array(ps)
def fit(m,tr,va,epochs=3,pos_weight=None,lr=1e-3):
    m.to(DEVICE);opt=torch.optim.AdamW(m.parameters(),lr=lr);pw=torch.tensor([pos_weight],device=DEVICE) if pos_weight else None;lossfn=nn.BCEWithLogitsLoss(pos_weight=pw);best=None;best_ap=-1
    for ep in range(epochs):
        m.train()
        for x,y in tr:
            x,y=x.to(DEVICE),y.to(DEVICE);opt.zero_grad(set_to_none=True);loss=lossfn(m(x),y);loss.backward();opt.step()
        yy,pp=evaluate(m,va);mm=mets(yy,pp);print(ep+1,mm)
        if mm['pr_auc']>best_ap:best_ap=mm['pr_auc'];best={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(best);return m
pos_weight=float((train_df.label==0).sum()/max(1,(train_df.label==1).sum()))
oh=fit(OneHotMCNN(),ohtr,ohva,epochs=2 if FAST_MODE else 8,pos_weight=pos_weight)

## 8. Load a small ESM-2 for the classroom pipeline

The published workflow can use larger PLM features. Here we use 8M ESM-2 so the entire embedding step is visible and fast.

In [ ]:
esm_model,alphabet=esm.pretrained.esm2_t6_8M_UR50D();esm_model=esm_model.to(DEVICE).eval();converter=alphabet.get_batch_converter();ESM_DIM=320
for p in esm_model.parameters():p.requires_grad=False

## 9. Inspect one peptide's per-residue embedding shape

In [ ]:
seq=train_df.sequence.iloc[0]
labels,strs,tokens=converter([('demo',seq)]);tokens=tokens.to(DEVICE)
with torch.no_grad():rep=esm_model(tokens,repr_layers=[6])['representations'][6][0,1:len(seq)+1]
print('sequence',seq,'length',len(seq),'embedding',rep.shape)

## 10. Cache ESM embeddings for the teaching subset

In [ ]:
@torch.no_grad()
def embed_many(seqs,batch_size=32):
    out=[]
    for st in range(0,len(seqs),batch_size):
        batch=[(str(i),s) for i,s in enumerate(seqs[st:st+batch_size])];_,strs,tok=converter(batch);tok=tok.to(DEVICE);r=esm_model(tok,repr_layers=[6])['representations'][6]
        for j,s in enumerate(strs):
            z=torch.zeros(MAX_LEN,ESM_DIM);z[:min(MAX_LEN,len(s))]=r[j,1:len(s)+1][:MAX_LEN].cpu();out.append(z)
    return torch.stack(out)
XtrE=embed_many(train_df.sequence.tolist());XvaE=embed_many(val_df.sequence.tolist());XteE=embed_many(test_df.sequence.tolist())
print(XtrE.shape)

## 11. Compare mean-pooled ESM vs ESM + MCNN

This separates the value of PLM pretraining from the value of downstream local pattern extraction.

In [ ]:
def meanpool(X,seqs):
    arr=[]
    for x,s in zip(X,seqs):arr.append(x[:min(len(s),MAX_LEN)].mean(0).numpy())
    return np.stack(arr)
mptr=meanpool(XtrE,train_df.sequence);mpva=meanpool(XvaE,val_df.sequence)
plr=LogisticRegression(max_iter=1000,class_weight='balanced').fit(mptr,train_df.label);pp=plr.predict_proba(mpva)[:,1]
print('mean ESM validation',mets(np.array(val_df.label),pp))

In [ ]:
class TensorDS(Dataset):
    def __init__(self,X,y):self.X=X;self.y=torch.tensor(np.asarray(y),dtype=torch.float32)
    def __len__(self):return len(self.y)
    def __getitem__(self,i):return self.X[i],self.y[i]
estr=DataLoader(TensorDS(XtrE,train_df.label),64,shuffle=True);esva=DataLoader(TensorDS(XvaE,val_df.label),64);este=DataLoader(TensorDS(XteE,test_df.label),64)
class ESM_MCNN(nn.Module):
    def __init__(self,windows=(2,4,8,16),filters=64,drop=.3):
        super().__init__();self.convs=nn.ModuleList([nn.Conv1d(ESM_DIM,filters,k) for k in windows]);self.drop=nn.Dropout(drop);self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,x):
        x=x.transpose(1,2);feats=[F.adaptive_max_pool1d(F.relu(c(x)),1).squeeze(-1) for c in self.convs];return self.fc(self.drop(torch.cat(feats,1))).squeeze(1)
esm_mcnn=fit(ESM_MCNN(),estr,esva,epochs=2 if FAST_MODE else 8,pos_weight=pos_weight)

## 12. Controlled window-size experiment

In [ ]:
rows=[]
for w in [2,4,8,16]:
    m=fit(ESM_MCNN(windows=(w,),filters=64),estr,esva,epochs=1 if FAST_MODE else 5,pos_weight=pos_weight)
    y,p=evaluate(m,esva);rows.append({'window':w,**mets(y,p)})
display(pd.DataFrame(rows))

## 13. Weighted vs unweighted loss

In [ ]:
unweighted=fit(ESM_MCNN(),estr,esva,epochs=1 if FAST_MODE else 5,pos_weight=None)
yw,pw=evaluate(esm_mcnn,este);yu,pu=evaluate(unweighted,este)
display(pd.DataFrame([{'loss':'weighted',**mets(yw,pw)},{'loss':'unweighted',**mets(yu,pu)}]))

## 14. Final independent-test comparison

For paper reproduction, replace the teaching split with the repository's exact test set before interpreting these numbers scientifically.

In [ ]:
yo,po=evaluate(oh,ohte)
mpte=meanpool(XteE,test_df.sequence);pm=plr.predict_proba(mpte)[:,1]
display(pd.DataFrame([
 {'model':'AA composition + logistic regression',**mets(np.array(test_df.label),lr.predict_proba(np.stack([comp(s) for s in test_df.sequence]))[:,1])},
 {'model':'one-hot + MCNN',**mets(yo,po)},
 {'model':'mean ESM + logistic regression',**mets(np.array(test_df.label),pm)},
 {'model':'ESM + MCNN',**mets(yw,pw)}]))

## 15. Error analysis

Inspect false positives and false negatives separately. In a rare-positive problem, the biological meaning of these two error types is very different.

In [ ]:
err=pd.DataFrame({'sequence':test_df.sequence.values,'label':test_df.label.values,'p_neo':pw});err['pred']=(err.p_neo>=.5).astype(int)
print('false negatives');display(err[(err.label==1)&(err.pred==0)].sort_values('p_neo').head(15))
print('false positives');display(err[(err.label==0)&(err.pred==1)].sort_values('p_neo',ascending=False).head(15))

# From classroom DeepNeoAG-Lite to research-grade reproduction

1. Use the public repository's exact train/test files.
2. Preserve the <30% similarity-control procedure.
3. Use the documented ProtTrans/TAPE/ESM features for direct comparison with the paper.
4. Use cross-validation for hyperparameter selection and keep the independent test fixed.
5. Repeat final models with multiple seeds.
6. Report PR-AUC, MCC, precision, recall/sensitivity, specificity, ROC-AUC, and TP/TN/FP/FN.
7. Interpret local windows cautiously; predictive patterns are hypotheses, not mechanisms.
8. Add external cohorts before claiming broad generalization.